<a href="https://colab.research.google.com/github/sh-reps/tcp/blob/main/LEX_and_YACC_Compiler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LEX and YACC Compiler in Colab

Drawbacks:
* Regular interrupts (Ctrl+D, Ctrl+C) for shell won't work in Colab while inputting for program.
<br>Workaround: Store your inputs in a txt file and pass it to the program.

In [2]:
#@title Install *prerqeuisites* (run this cell first to work on LEX/YACC)
!sudo apt install flex bison

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison-doc flex-doc
The following NEW packages will be installed:
  bison flex libfl-dev libfl2
0 upgraded, 4 newly installed, 0 to remove and 0 not upgraded.
Need to get 1,066 kB of archives.
After this operation, 3,583 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 flex amd64 2.6.4-8.2build1 [301 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 bison amd64 2:3.8.2+dfsg-1ubuntu0.24.04.1 [749 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/main amd64 libfl2 amd64 2.6.4-8.2build1 [9,768 B]
Get:4 http://archive.ubuntu.com/ubuntu noble/main amd64 libfl-dev amd64 2.6.4-8.2build1 [6,250 B]
Fetched 1,066 kB in 55s (19.4 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dia

## Lex only

In [5]:
%%writefile program.l
%{
#include <stdio.h>
int chars = 0, words = 0;
%}

%%
[ \t\n]+    { chars += yyleng; }
[^ \t\n]+   { chars += yyleng; words++; }
%%

int main()
{
    printf("Enter text:\n");
    yylex();

    printf("Characters = %d\n", chars);
    printf("Words = %d\n", words);

    return 0;
}

int yywrap()
{
    return 1;
}

Overwriting program.l


if you want to use at txt as an input

In [8]:
%%writefile program.txt

This is a sample file.

Writing program.txt


In [9]:
#@title Shell Execution (you can rewrite the commands as per your need, eg. if you want to include a file as an input)
%%shell

lex program.l
gcc lex.yy.c -o program -ll
./program < program.txt

Enter text:
Characters = 24
Words = 5


## Lex and Yacc combined

In [25]:
%%writefile program.y
%{
#include <stdio.h>
int yylex();
void yyerror(char *s);
%}

%token KEYWORD ID

%%
/* Allow a list of words, ignoring newlines or extra spaces */
input: /* empty */
     | input word
     ;

word : KEYWORD  { printf("Keyword\n"); }
     | ID       { printf("Valid Identifier\n"); }
     ;
%%

void yyerror(char *s)
{
    printf("Invalid Input\n");
}

int main()
{
    printf("Enter a word:\n");
    yyparse();
    return 0;
}

Overwriting program.y


In [26]:
%%writefile program.l
%{
#include "y.tab.h"
#include <string.h>
%}

%%
"int"|"float"|"char"|"if"|"else"|"while" {
    return KEYWORD;
}

[a-zA-Z_][a-zA-Z0-9_]* {
    return ID;
}

[ \t\n]+    ;
.           { return yytext[0]; }
%%

Overwriting program.l


if you want to use at txt as an input

In [27]:
%%writefile program.txt

int idea

Overwriting program.txt


In [28]:
#@title Shell Execution (you can rewrite the commands as per your need, eg. if you want to include a file as an input)
%%shell

yacc -d program.y
lex program.l
gcc y.tab.c lex.yy.c -o program -ll
./program <program.txt

Enter a word:
Keyword
Valid Identifier
